                # Step 3 - QC and Preprocessing

                This notebook documents the quality-control checks and the preprocessing logic
                that should remain leakage-safe. The processed matrix must be created first by
                running `02_probe_mapping_and_gene_matrix.ipynb`.
                


In [ ]:
import sys; sys.path.insert(0, '../src')
                import pandas as pd
                from sklearn.feature_selection import VarianceThreshold
                from sklearn.impute import SimpleImputer
                from sklearn.pipeline import Pipeline
                from sklearn.preprocessing import StandardScaler

                from asd_pipeline_utils import load_analysis_frame, split_features_and_metadata

                final_df = load_analysis_frame()
                X, meta = split_features_and_metadata(final_df)

                print("Analysis frame shape:", final_df.shape)
                print("Feature matrix shape:", X.shape)
                print("Metadata columns:", list(meta.columns))
                


In [ ]:
                # Inspect the missingness profile and class imbalance before model fitting.
                missing_rate = X.isna().mean()
                variance = X.var(axis=0)

                print("Columns with any missing values:", int((missing_rate > 0).sum()))
                print("Median gene variance:", float(variance.median()))
                print("Top class counts:")
                print(meta["diagnosis"].value_counts())

                print("\nDiagnosis by platform:")
                print(pd.crosstab(meta["diagnosis"], meta["platform_id"]))
                


In [ ]:
                # Define preprocessing as a pipeline so that fitting stays inside cross-validation.
                preprocessing = Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("var", VarianceThreshold(threshold=0.0)),
                        ("scaler", StandardScaler()),
                    ]
                )

                preprocessing
                
